# Projekt, 3. část: příprava dat a jejich popisná charakteristika
**Tým xkaska02**

**Řešitelé:**
- Martin Burian (xburiam00)
- Karel Kaska (xkaska02)
- David Kvaček (xkvace00)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

## 1. Načtení a základní předzpracování dat

In [ ]:
df = pd.read_table("data.tsv", names=['url', 'name', 'price', 'brand', 'model', 
                                       'color', 'num_of_doors', 'num_of_seats', 
                                       'year_of_manufacture'])

# Konverze ceny na float
df['price'] = df['price'].str.replace(" ", "").str.replace(",", ".").astype("float")

print(f"Dataset obsahuje {len(df)} řádků a {len(df.columns)} sloupců")
df.head()

## 2. Prozkoumání datové sady a atributů

In [ ]:
df.info()

In [ ]:
print("Numerické atributy:")
display(df.describe())

print("\nKategorické atributy:")
display(df.describe(include=['object']))

In [ ]:
for col in df.columns:
    print(f"\n{col}:")
    print(f"  Typ: {df[col].dtype}, Unikátní: {df[col].nunique()}, Chybějící: {df[col].isna().sum()}")
    
    if df[col].dtype == 'object':
        print(f"  Top 5: {df[col].value_counts().head(5).to_dict()}")
    else:
        print(f"  Min: {df[col].min()}, Max: {df[col].max()}, Průměr: {df[col].mean():.2f}")

## 3. Vizualizace rozložení hodnot

In [ ]:
# Graf 1: Histogram a boxplot ceny
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(df['price'].dropna(), bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Cena')
axes[0].set_ylabel('Četnost')
axes[0].set_title('Histogram cen vozidel')
axes[0].grid(True, alpha=0.3)

axes[1].boxplot(df['price'].dropna())
axes[1].set_ylabel('Cena')
axes[1].set_title('Boxplot cen')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Medián: {df['price'].median():.0f}, Průměr: {df['price'].mean():.0f}")
print("Rozložení je pravostranně zešikmené - většina aut má nižší cenu.")

In [ ]:
# Graf 2: Scatter plot - rok vs cena
plt.figure(figsize=(12, 6))
plt.scatter(df['year_of_manufacture'], df['price'], alpha=0.5, s=30)
plt.xlabel('Rok výroby')
plt.ylabel('Cena')
plt.title('Vztah mezi rokem výroby a cenou')
plt.grid(True, alpha=0.3)

z = np.polyfit(df['year_of_manufacture'].dropna(), df['price'].dropna(), 1)
p = np.poly1d(z)
plt.plot(df['year_of_manufacture'].sort_values(), 
         p(df['year_of_manufacture'].sort_values()), 
         "r--", alpha=0.8, linewidth=2, label='Trend')
plt.legend()
plt.show()

print("Novější vozidla jsou dražší - pozitivní korelace.")

In [ ]:
# Graf 3: Boxplot cen podle značky
top_brands = df['brand'].value_counts().head(15).index
df_top = df[df['brand'].isin(top_brands)]

fig, ax = plt.subplots(figsize=(16, 6))
sns.boxplot(x="brand", y="price", data=df_top, ax=ax)
plt.xticks(rotation=45, ha='right')
plt.xlabel('Značka')
plt.ylabel('Cena')
plt.title('Rozložení cen podle značky (top 15)')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("Výrazné rozdíly mezi značkami. Prémiové značky mají vyšší mediánové ceny.")

In [ ]:
# Graf 4: Violin plot - počet dveří
plt.figure(figsize=(12, 6))
sns.violinplot(x="num_of_doors", y="price", data=df)
plt.xlabel('Počet dveří')
plt.ylabel('Cena')
plt.title('Rozložení cen podle počtu dveří')
plt.grid(True, alpha=0.3, axis='y')
plt.show()

print("Šířka houslí ukazuje hustotu rozložení - 5dveřová auta mají širší cenové rozpětí.")

In [ ]:
# Graf 5: Heatmap - značka vs barva
top_brands = df['brand'].value_counts().head(10).index
top_colors = df['color'].value_counts().head(8).index

df_pivot = df[df['brand'].isin(top_brands) & df['color'].isin(top_colors)].pivot_table(
    values='price', index='brand', columns='color', aggfunc='mean'
)

plt.figure(figsize=(12, 8))
sns.heatmap(df_pivot, annot=True, fmt='.0f', cmap='YlOrRd', cbar_kws={'label': 'Průměrná cena'})
plt.title('Průměrné ceny podle značky a barvy')
plt.xlabel('Barva')
plt.ylabel('Značka')
plt.tight_layout()
plt.show()

print("Některé kombinace značky a barvy mají výrazně odlišné ceny.")

In [ ]:
# Graf 6: Distribuce podle roku výroby
year_counts = df['year_of_manufacture'].value_counts().sort_index()

plt.figure(figsize=(14, 6))
plt.bar(year_counts.index, year_counts.values, edgecolor='black', alpha=0.7)
plt.xlabel('Rok výroby')
plt.ylabel('Počet vozidel')
plt.title('Distribuce vozidel podle roku výroby')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 4. Detekce odlehlých hodnot

In [ ]:
# Metoda 1: IQR
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower) | (data[column] > upper)]
    return outliers, lower, upper

outliers_price, lower_p, upper_p = detect_outliers_iqr(df, 'price')
print(f"IQR metoda - Cena:")
print(f"  Rozsah: [{lower_p:.0f}, {upper_p:.0f}]")
print(f"  Outliers: {len(outliers_price)} ({len(outliers_price)/len(df)*100:.1f}%)")

outliers_year, lower_y, upper_y = detect_outliers_iqr(df, 'year_of_manufacture')
print(f"\nIQR metoda - Rok výroby:")
print(f"  Rozsah: [{lower_y:.0f}, {upper_y:.0f}]")
print(f"  Outliers: {len(outliers_year)} ({len(outliers_year)/len(df)*100:.1f}%)")

In [ ]:
# Metoda 2: Z-score
def detect_outliers_zscore(data, column, threshold=3):
    z_scores = np.abs(stats.zscore(data[column].dropna()))
    outliers_idx = np.where(z_scores > threshold)[0]
    return data.iloc[outliers_idx]

outliers_z = detect_outliers_zscore(df, 'price')
print(f"Z-score metoda (|z| > 3):")
print(f"  Outliers v ceně: {len(outliers_z)} ({len(outliers_z)/len(df)*100:.1f}%)")

In [ ]:
# Metoda 3: Modified Z-score (MAD)
def detect_outliers_mad(data, column, threshold=3.5):
    median = data[column].median()
    mad = np.median(np.abs(data[column] - median))
    modified_z = 0.6745 * (data[column] - median) / mad
    return data[np.abs(modified_z) > threshold]

outliers_mad = detect_outliers_mad(df, 'price')
print(f"Modified Z-score metoda:")
print(f"  Outliers v ceně: {len(outliers_mad)} ({len(outliers_mad)/len(df)*100:.1f}%)")
print(f"\nRůzné metody identifikují různý počet outlierů. MAD je robustnější.")

## 5. Analýza chybějících hodnot

In [ ]:
missing = pd.DataFrame({
    'Počet': df.isnull().sum(),
    'Procento': (df.isnull().sum() / len(df)) * 100
})
missing = missing[missing['Počet'] > 0]

if len(missing) == 0:
    print("Dataset neobsahuje chybějící hodnoty.")
else:
    print("Chybějící hodnoty:")
    display(missing)
    
    rows_missing = df[df.isnull().any(axis=1)]
    print(f"\nŘádky s chybějícími hodnotami: {len(rows_missing)} ({len(rows_missing)/len(df)*100:.1f}%)")
    
    plt.figure(figsize=(12, 6))
    sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
    plt.title('Heatmap chybějících hodnot')
    plt.show()

## 6. Korelační analýza

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Korelační matice numerických atributů')
plt.tight_layout()
plt.show()

print("\nNejvýznamnější korelace:")
print(corr_matrix['price'].sort_values(ascending=False))

---
# Příprava datových sad

## Odstranění irelevantních atributů a outlierů

In [ ]:
# Odstranění url a name (irelevantní pro predikci ceny)
df_clean = df.drop(columns=['url', 'name'])

# Odstranění outlierů pomocí IQR
outliers_mask = outliers_price.index.union(outliers_year.index)
df_clean = df_clean[~df_clean.index.isin(outliers_mask)]

print(f"Původní: {len(df)} řádků")
print(f"Po čištění: {len(df_clean)} řádků")
print(f"Odstraněno: {len(df) - len(df_clean)} řádků")

## Varianta 1: Kategorická dataset

In [ ]:
df_v1 = df_clean.copy()

# Ošetření chybějících hodnot
# Metoda 1: Odstranění řádků (ukázka)
df_v1_drop = df_v1.dropna()
print(f"Metoda 1 - Odstranění řádků: {len(df_v1)} -> {len(df_v1_drop)}")

# Metoda 2: Imputace (použijeme tuto)
for col in df_v1.select_dtypes(include=['object']).columns:
    if df_v1[col].isnull().sum() > 0:
        df_v1[col].fillna(df_v1[col].mode()[0], inplace=True)

for col in df_v1.select_dtypes(include=[np.number]).columns:
    if df_v1[col].isnull().sum() > 0:
        df_v1[col].fillna(df_v1[col].median(), inplace=True)

print("Metoda 2 - Imputace: kompletní")

In [ ]:
# Diskretizace numerických atributů
df_v1['price_cat'] = pd.cut(df_v1['price'], 
                             bins=[0, 50000, 100000, 200000, float('inf')],
                             labels=['levné', 'střední', 'drahé', 'luxusní'])

df_v1['age'] = 2024 - df_v1['year_of_manufacture']
df_v1['age_cat'] = pd.cut(df_v1['age'],
                           bins=[-1, 3, 7, 15, float('inf')],
                           labels=['velmi nové', 'nové', 'střední', 'staré'])

df_v1['doors_cat'] = df_v1['num_of_doors'].astype(str) + '_doors'
df_v1['seats_cat'] = df_v1['num_of_seats'].astype(str) + '_seats'

# Finální výběr sloupců
df_variant1 = df_v1[['brand', 'model', 'color', 'doors_cat', 'seats_cat', 
                      'age_cat', 'price_cat']].copy()

print(f"Varianta 1: {df_variant1.shape}")
df_variant1.head()

In [ ]:
df_variant1.head(50).to_csv('cat.csv', index=False)
print("Uloženo: cat.csv")

## Varianta 2: Numerická dataset

In [ ]:
df_v2 = df_clean.copy()

# Ošetření chybějících hodnot stejně jako u varianty 1
for col in df_v2.select_dtypes(include=['object']).columns:
    if df_v2[col].isnull().sum() > 0:
        df_v2[col].fillna(df_v2[col].mode()[0], inplace=True)

for col in df_v2.select_dtypes(include=[np.number]).columns:
    if df_v2[col].isnull().sum() > 0:
        df_v2[col].fillna(df_v2[col].median(), inplace=True)

print("Chybějící hodnoty ošetřeny")

In [ ]:
# Transformace kategorických atributů
# Label Encoding pro brand a model
le_brand = LabelEncoder()
le_model = LabelEncoder()
df_v2['brand_enc'] = le_brand.fit_transform(df_v2['brand'])
df_v2['model_enc'] = le_model.fit_transform(df_v2['model'])

# One-Hot Encoding pro color
color_dummies = pd.get_dummies(df_v2['color'], prefix='color')
df_v2 = pd.concat([df_v2, color_dummies], axis=1)

# Odstranění původních kategorických sloupců
df_v2 = df_v2.drop(['brand', 'model', 'color'], axis=1)

print(f"Encoded shape: {df_v2.shape}")

In [ ]:
# Normalizace do <0,1>
cols_to_norm = ['price', 'num_of_doors', 'num_of_seats', 'year_of_manufacture',
                'brand_enc', 'model_enc']

scaler = MinMaxScaler()
df_v2[cols_to_norm] = scaler.fit_transform(df_v2[cols_to_norm])

df_variant2 = df_v2.copy()

print(f"Varianta 2: {df_variant2.shape}")
print(f"Všechny hodnoty v <0,1>: {(df_variant2.min() >= 0).all() and (df_variant2.max() <= 1).all()}")
df_variant2.head()

In [ ]:
df_variant2.head(50).to_csv('num.csv', index=False)
print("Uloženo: num.csv")

## Shrnutí

**Varianta 1:** Kategorická data pro algoritmy jako Decision Trees, Naive Bayes  
**Varianta 2:** Numerická data normalizovaná do <0,1> pro Neural Networks, SVM, k-NN